# 14 — Load & Parse Deep Analysis Dataframes

Load df1 (weights), df2 (eigenvalues), df3 (topology) from all completed experiments.

In [1]:
import numpy as np, pandas as pd, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

NB = Path('/home/aymen/Documents/GitHub/Federated-Continual-learning-/New/notebooks_sandbox')
DEEP = NB / 'paper_results' / 'deep_analysis'

# Discover all completed experiments
exps = sorted([d for d in DEEP.iterdir() if d.is_dir() and (d/'df1_weights.pkl').exists()])
print(f'Completed experiments: {len(exps)}')
for e in exps:
    print(f'  {e.name}')

Completed experiments: 102
  tiny_overlap0_AUTO
  tiny_overlap0_DiffPers
  tiny_overlap0_FFT
  tiny_overlap0_FFT_0.1xMelSpec
  tiny_overlap0_FIM
  tiny_overlap0_LWLN
  tiny_overlap0_LW_DiffPers
  tiny_overlap0_LW_FFT
  tiny_overlap0_LW_FFT_0.1xLW_MelSpec
  tiny_overlap0_LW_MAE
  tiny_overlap0_LW_MAE_0.1xLW_DiffPers
  tiny_overlap0_LW_MAPE
  tiny_overlap0_LW_MAPE_0.1xLW_JS
  tiny_overlap0_LW_MSE
  tiny_overlap0_LW_MSE_0.01xLW_PersLandscape
  tiny_overlap0_LW_MSE_0.05xLW_Frobenius
  tiny_overlap0_LW_MSE_0.05xLW_RTD
  tiny_overlap0_LW_MSE_0.1xLW_DiffPers
  tiny_overlap0_LW_MSE_0.1xLW_LogNorm
  tiny_overlap0_LW_RTD
  tiny_overlap0_LW_Sinkhorn
  tiny_overlap0_MAE
  tiny_overlap0_MAE_0.05xRTD
  tiny_overlap0_MAE_0.1xDiffPers
  tiny_overlap0_MAPE
  tiny_overlap0_MAPE_0.1xJS
  tiny_overlap0_MSE
  tiny_overlap0_MSE_0.01xPersLandscape
  tiny_overlap0_MSE_0.05xFrobenius
  tiny_overlap0_MSE_0.05xRTD
  tiny_overlap0_MSE_0.15xSinkhorn
  tiny_overlap0_MSE_0.1xDiffPers
  tiny_overlap0_MSE_0.1xLogNorm


## 1. Load Single Experiment

In [2]:
# Pick an experiment (change this)
EXP_NAME = exps[0].name if exps else 'tiny_overlap0_MSE'
print(f'Loading: {EXP_NAME}')

exp_dir = DEEP / EXP_NAME
df1 = pd.read_pickle(exp_dir / 'df1_weights.pkl')
df2 = pd.read_pickle(exp_dir / 'df2_eigenvalues.pkl')
df3 = pd.read_pickle(exp_dir / 'df3_topology.pkl')
with open(exp_dir / 'selected_scales.json') as f:
    scales = json.load(f)

print(f'DF1 (weights):     {df1.shape}  cols: {list(df1.columns)}')
print(f'DF2 (eigenvalues): {df2.shape}  cols: {list(df2.columns)}')
print(f'DF3 (topology):    {df3.shape}  cols: {list(df3.columns)}')
print(f'Scales: {scales}')

Loading: tiny_overlap0_AUTO
DF1 (weights):     (100, 16)  cols: ['sample', 'loss', 'overlap', 'i1', 'i2', 'pred', 'gt', 'fn1', 'fn2', 'fn3', 'fn4', 'fn5', 'cnn_acc_init', 'cnn_acc_final', 'cnn_acc_per_epoch', 'cnn_loss_per_epoch']
DF2 (eigenvalues): (900, 29)  cols: ['sample', 'weight_type', 'loss', 'overlap', 'eig_conv1', 'mp_min_conv1', 'mp_max_conv1', 'mp_peak_conv1', 'frac_outside_mp_conv1', 'eig_conv2', 'mp_min_conv2', 'mp_max_conv2', 'mp_peak_conv2', 'frac_outside_mp_conv2', 'eig_conv3', 'mp_min_conv3', 'mp_max_conv3', 'mp_peak_conv3', 'frac_outside_mp_conv3', 'eig_fc1', 'mp_min_fc1', 'mp_max_fc1', 'mp_peak_fc1', 'frac_outside_mp_fc1', 'eig_fc2', 'mp_min_fc2', 'mp_max_fc2', 'mp_peak_fc2', 'frac_outside_mp_fc2']
DF3 (topology):    (900, 19)  cols: ['sample', 'weight_type', 'loss', 'overlap', 'betti_0', 'betti_1', 'h0_n_features', 'h0_total_persistence', 'h0_persistence_entropy', 'h0_max_lifetime', 'h0_mean_lifetime', 'h1_n_features', 'h1_total_persistence', 'h1_persistence_entropy

## 2. DF1 — Weights

In [3]:
# Scalar columns
scalar_cols = [c for c in df1.columns if not isinstance(df1[c].iloc[0], (np.ndarray, list))]
print('Scalar columns:', scalar_cols)
print()
df1[scalar_cols].describe().round(2)

Scalar columns: ['sample', 'loss', 'overlap', 'cnn_acc_init', 'cnn_acc_final']



,sample,overlap,cnn_acc_init,cnn_acc_final
count,100.00,100.0,100.00,100.00
mean,49.50,0.0,5.81,76.26
std,29.01,0.0,4.15,14.00
min,0.00,0.0,2.09,52.28
25%,24.75,0.0,2.38,63.48
50%,49.50,0.0,2.69,77.28
75%,74.25,0.0,10.31,92.27
max,99.00,0.0,11.94,96.04


In [4]:
# CNN accuracy per epoch
if 'cnn_acc_per_epoch' in df1.columns:
    print('CNN accuracy trajectories:')
    for i in range(min(5, len(df1))):
        accs = df1.iloc[i]['cnn_acc_per_epoch']
        print(f'  Sample {i}: {[f"{a:.1f}" for a in accs]}')

if 'cnn_loss_per_epoch' in df1.columns:
    print('\nCNN loss trajectories:')
    for i in range(min(5, len(df1))):
        losses = df1.iloc[i]['cnn_loss_per_epoch']
        print(f'  Sample {i}: {[f"{l:.4f}" for l in losses]}')

CNN accuracy trajectories:
  Sample 0: ['2.1', '56.1', '61.2', '62.7', '62.8', '64.3']
  Sample 1: ['2.7', '62.8', '89.0', '92.0', '93.4', '94.1']
  Sample 2: ['11.6', '38.6', '70.4', '76.1', '76.1', '76.8']
  Sample 3: ['2.1', '49.2', '61.9', '62.4', '63.2', '62.8']
  Sample 4: ['2.1', '47.0', '61.8', '62.5', '63.6', '64.0']

CNN loss trajectories:
  Sample 0: ['1.5219', '0.4819', '0.3116', '0.2510', '0.2174']
  Sample 1: ['1.6061', '0.5705', '0.3033', '0.2427', '0.2092']
  Sample 2: ['1.7825', '0.7435', '0.3231', '0.2318', '0.1848']
  Sample 3: ['1.6223', '0.5286', '0.3125', '0.2523', '0.2217']
  Sample 4: ['1.6553', '0.5906', '0.3260', '0.2364', '0.1949']


In [5]:
# Weight vector shapes
wt_cols = ['i1','i2','pred','gt'] + [c for c in df1.columns if c.startswith('fn')]
print('Weight vector columns:', [c for c in wt_cols if c in df1.columns])
for c in wt_cols:
    if c in df1.columns:
        v = df1.iloc[0][c]
        if isinstance(v, np.ndarray):
            print(f'  {c}: shape={v.shape} dtype={v.dtype} range=[{v.min():.4f}, {v.max():.4f}]')

Weight vector columns: ['i1', 'i2', 'pred', 'gt', 'fn1', 'fn2', 'fn3', 'fn4', 'fn5']
  i1: shape=(2464,) dtype=float32 range=[-1.9908, 1.6892]
  i2: shape=(2464,) dtype=float32 range=[-1.7675, 1.9776]
  pred: shape=(2464,) dtype=float32 range=[-3.2517, 3.3438]
  gt: shape=(2464,) dtype=float32 range=[-1.9356, 2.2763]
  fn1: shape=(2464,) dtype=float32 range=[-3.1015, 3.5287]
  fn2: shape=(2464,) dtype=float32 range=[-3.0846, 3.6501]
  fn3: shape=(2464,) dtype=float32 range=[-3.0866, 3.7225]
  fn4: shape=(2464,) dtype=float32 range=[-3.0820, 3.7809]
  fn5: shape=(2464,) dtype=float32 range=[-3.0443, 3.8265]


## 3. DF2 — Eigenvalues

In [6]:
print('Weight types:', df2['weight_type'].unique())
print('Samples:', df2['sample'].unique())
print()
# Scalar columns
eig_scalar = [c for c in df2.columns if not c.startswith('eig_') and c not in ['sample','weight_type','loss','overlap']]
print('MP bounds columns:', eig_scalar)
print()
# Show eigenvalue shapes
LAYER_NAMES = ['conv1','conv2','conv3','fc1','fc2']
for ln in LAYER_NAMES:
    col = f'eig_{ln}'
    if col in df2.columns:
        v = df2.iloc[0][col]
        print(f'  {col}: shape={np.array(v).shape} range=[{np.array(v).min():.6f}, {np.array(v).max():.6f}]')

Weight types: ['i1' 'i2' 'pred' 'gt' 'fn1' 'fn2' 'fn3' 'fn4' 'fn5']
Samples: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]

MP bounds columns: ['mp_min_conv1', 'mp_max_conv1', 'mp_peak_conv1', 'frac_outside_mp_conv1', 'mp_min_conv2', 'mp_max_conv2', 'mp_peak_conv2', 'frac_outside_mp_conv2', 'mp_min_conv3', 'mp_max_conv3', 'mp_peak_conv3', 'frac_outside_mp_conv3', 'mp_min_fc1', 'mp_max_fc1', 'mp_peak_fc1', 'frac_outside_mp_fc1', 'mp_min_fc2', 'mp_max_fc2', 'mp_peak_fc2', 'frac_outside_mp_fc2']

  eig_conv1: shape=(8,) range=[1.161524, 7.008046]
  eig_conv2: shape=(6,) range=[2.671492, 6.411247]
  eig_conv3: shape=(4,) range=[1.435009, 3.229642]
  eig_fc1: shape=(20,) range=[0.240847, 13.998657]
  eig_fc2: shape=(10,) range=[0.51410

In [ ]:
# Summary of fraction outside MP bulk
frac_cols = [c for c in df2.columns if c.startswith('frac_')]
print('Fraction outside MP bulk (mean across samples):')
for wt in df2['weight_type'].unique():
    sub = df2[df2['weight_type'] == wt]
    vals = {c: sub[c].mean() for c in frac_cols}
    parts = [f"{c.replace('frac_outside_mp_', '')}={v:.3f}" for c, v in vals.items()]
    print(f'  {wt:5s}: {"  ".join(parts)}')

SyntaxError: f-string: f-string: unmatched '(' (320989363.py, line 7)

## 4. DF3 — Topology

In [8]:
print('Weight types:', df3['weight_type'].unique())
print('Columns:', list(df3.columns))
print()
# Show topology summary per weight type
topo_cols = [c for c in df3.columns if c not in ['sample','weight_type','loss','overlap']]
print('Topology columns:', topo_cols)
print()
for wt in df3['weight_type'].unique():
    sub = df3[df3['weight_type'] == wt]
    print(f'\n  {wt}:')
    for c in topo_cols:
        vals = sub[c].dropna()
        if len(vals) > 0:
            print(f'    {c:30s}  mean={vals.mean():.4f}  std={vals.std():.4f}  range=[{vals.min():.4f}, {vals.max():.4f}]')

Weight types: ['i1' 'i2' 'pred' 'gt' 'fn1' 'fn2' 'fn3' 'fn4' 'fn5']
Columns: ['sample', 'weight_type', 'loss', 'overlap', 'betti_0', 'betti_1', 'h0_n_features', 'h0_total_persistence', 'h0_persistence_entropy', 'h0_max_lifetime', 'h0_mean_lifetime', 'h1_n_features', 'h1_total_persistence', 'h1_persistence_entropy', 'h1_max_lifetime', 'h1_mean_lifetime', 'mpdist_optimal', 'mpdist_breakpoint', 'mpdist_meta']

Topology columns: ['betti_0', 'betti_1', 'h0_n_features', 'h0_total_persistence', 'h0_persistence_entropy', 'h0_max_lifetime', 'h0_mean_lifetime', 'h1_n_features', 'h1_total_persistence', 'h1_persistence_entropy', 'h1_max_lifetime', 'h1_mean_lifetime', 'mpdist_optimal', 'mpdist_breakpoint', 'mpdist_meta']


  i1:
    betti_0                         mean=161.2700  std=4.3295  range=[146.0000, 175.0000]
    betti_1                         mean=0.0000  std=0.0000  range=[0.0000, 0.0000]
    h0_n_features                   mean=160.2700  std=4.3295  range=[145.0000, 174.0000]
    h0_tot

## 5. Load ALL Experiments (combined)

In [9]:
all_df1, all_df2, all_df3 = [], [], []

for exp_dir in exps:
    d1 = pd.read_pickle(exp_dir / 'df1_weights.pkl')
    d2 = pd.read_pickle(exp_dir / 'df2_eigenvalues.pkl')
    d3 = pd.read_pickle(exp_dir / 'df3_topology.pkl')
    d1['experiment'] = exp_dir.name
    d2['experiment'] = exp_dir.name
    d3['experiment'] = exp_dir.name
    all_df1.append(d1)
    all_df2.append(d2)
    all_df3.append(d3)

cdf1 = pd.concat(all_df1, ignore_index=True) if all_df1 else pd.DataFrame()
cdf2 = pd.concat(all_df2, ignore_index=True) if all_df2 else pd.DataFrame()
cdf3 = pd.concat(all_df3, ignore_index=True) if all_df3 else pd.DataFrame()

print(f'Combined DF1: {cdf1.shape}')
print(f'Combined DF2: {cdf2.shape}')
print(f'Combined DF3: {cdf3.shape}')

Combined DF1: (10200, 17)
Combined DF2: (91800, 30)
Combined DF3: (91800, 20)


In [10]:
# Per-experiment CNN accuracy summary
if len(cdf1) > 0:
    acc_summary = cdf1.groupby(['experiment','overlap','loss']).agg(
        mean_init=('cnn_acc_init','mean'),
        mean_final=('cnn_acc_final','mean'),
        std_final=('cnn_acc_final','std'),
        n_samples=('sample','count'),
    ).reset_index()
    acc_summary['improvement'] = acc_summary['mean_final'] - acc_summary['mean_init']
    acc_summary = acc_summary.sort_values(['overlap','mean_final'], ascending=[True, False])
    acc_summary = acc_summary.round(2)
    print(acc_summary.to_string(index=False))

                                experiment  overlap                         loss  mean_init  mean_final  std_final  n_samples  improvement
                    tiny_overlap0_Sinkhorn        0                     Sinkhorn      11.61       78.19      14.37        100        66.59
           tiny_overlap0_MSE_0.15xSinkhorn        0            MSE_0.15xSinkhorn      14.42       77.96      14.44        100        63.53
      tiny_overlap0_LW_MAE_0.1xLW_DiffPers        0       LW_MAE_0.1xLW_DiffPers       5.04       77.73      14.29        100        72.69
            tiny_overlap0_MAE_0.1xDiffPers        0             MAE_0.1xDiffPers       0.87       77.49      14.39        100        76.63
       tiny_overlap0_LW_MSE_0.1xLW_LogNorm        0        LW_MSE_0.1xLW_LogNorm      11.74       77.31      14.36        100        65.57
                         tiny_overlap0_MAE        0                          MAE       0.00       77.17      14.48        100        77.17
                 tiny_overl